In [17]:
import pypsa
import numpy as np
import pandas as pd
import pickle
import xarray as xr

In [45]:
horizon = 48
overlap = 0

# overlap 8 funktioniert super

nr_networks = 6

n_base = pypsa.Network('base_s_1__none_2035_lt.nc')

INFO:pypsa.io:Imported network base_s_1__none_2035_lt.nc has buses, carriers, generators, global_constraints, links, loads, storage_units, stores


In [12]:
from typing import Dict, Any

co2_prices = {
    "2020": 28,
    "2025": 70,
    "2030": 130,
    "2035": 190,
    "2040": 210,
    "2045": 530,
}

specific_emissions = {
    "oil" : 0.2571,
    "oil primary" : 0.2571,
    "gas" : 0.198, # OCGT
    "gas primary" : 0.198, # OCGT
    "coal" : 0.3361,
    "lignite" : 0.4069,
    "solid biomass" : 0.3667
}

def add_carbon_price(n, co2_price, specific_emissions):
        
    for carrier in specific_emissions.keys():
        n.generators.loc[n.generators.carrier == carrier, "marginal_cost"] += (
            co2_price * specific_emissions[carrier]
        )

def build_st_network(n, e_initial_h2 = 1e6):

    # alle hydrogen related stores
    stores = n.stores[n.stores.carrier.str.contains("H2", case=False, na=False)].index

    # marginal price für alle der mean
    for i in stores:
        n.stores.loc[i, 'marginal_cost'] = n_base.buses_t.marginal_price['DE0 0 H2'].mean()
        n.stores.loc[i, 'e_cyclic'] = False


    n.stores.loc['DE0 0 H2 Store-2035', 'e_initial'] = e_initial_h2
    n.stores.loc['DE0 0 H2 Store-2030', 'e_initial'] = 0
    n.stores.loc['DE0 0 H2 Store-2020', 'e_initial'] = 0

    n.optimize.fix_optimal_capacities()
    n.optimize.create_model()

    return n

def build_st_network_pertubated(n, e_initial_h2 = 1e6):

    # alle hydrogen related stores
    stores = n.stores[n.stores.carrier.str.contains("H2", case=False, na=False)].index

    # marginal price für alle der mean
    for i in stores:
        n.stores.loc[i, 'marginal_cost'] = n_base.buses_t.marginal_price['DE0 0 H2'].mean()
        n.stores.loc[i, 'e_cyclic'] = False


    n.stores.loc['DE0 0 H2 Store-2035', 'e_initial'] = e_initial_h2
    n.stores.loc['DE0 0 H2 Store-2030', 'e_initial'] = 0
    n.stores.loc['DE0 0 H2 Store-2020', 'e_initial'] = 0

    n.optimize.fix_optimal_capacities()
    n.generators.p_nom *= 1.10
    n.links.p_nom *= 1.10
    
    n.optimize.create_model()

    return n

In [13]:
# Display helper from Matplotlib (import is unused here but kept as provided)
from matplotlib.cbook import print_cycles


# Compute and display profit results for multiple networks and price scenarios
def print_profits(time_zone):
    # Table that will hold profits; rows = price cases + summary rows, columns = networks
    profit = pd.DataFrame()

    # Determine counts automatically from the provided collections
    num_networks = len(networks_deterministic)  # number of deterministic networks
    num_prices = len(price_list)                # number of price scenarios

    # Loop over each deterministic network and each price; compute profit per pair
    for net_idx, network in enumerate(networks_deterministic):
        for price_idx, price in enumerate(price_list):
            label = f"price {price_idx + 1}"
            profit.loc[label, f"n{net_idx}"] = get_profit(network, price, time_zone)

    # Add profits for the stochastic network across the same price scenarios
    for price_idx, price in enumerate(price_list):
        label = f"price {price_idx + 1}"
        profit.loc[label, "n_stochastic"] = get_profit(n_stochastic, price, time_zone)

    # Compute profits using the real (observed) price for all networks
    for net_idx, network in enumerate(networks_deterministic):
        profit.loc["real_price", f"n{net_idx}"] = get_profit(network, real_price, time_zone)

    # Real price for the stochastic network as well
    profit.loc["real_price", "n_stochastic"] = get_profit(n_stochastic, real_price, time_zone)

    # Units information for the printed table
    print("Profit in Mrd. €")

    # Row-wise mean across all synthetic price scenarios (exclude the 'real_price' row)
    for col in profit.columns:
        profit.loc["mean", col] = profit.drop('real_price')[col].mean()

    # Prepare formatting: round for display, then convert to billions of euros
    profit.round(2)
    profit = profit / 1e9

    # Helper to highlight the maximum value in each row (best network for that price case)
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: green' if v else '' for v in is_max]

    # Style the DataFrame: highlight row-wise maxima and format with two decimals
    styled_df = profit.style.apply(highlight_max, axis=1).format("{:.2f}")

    # Render the styled table in notebooks/interactive environments
    display(styled_df)

    # Return the numeric data (without styling) for further processing if needed
    return profit


In [74]:
def prepare_profit_optimization_network():
    m = pypsa.Network()

    # Load the base network (2035 scenario)
    n = pypsa.Network('base_s_1__none_2035_lt.nc')

    # Fix capacities from the base solution and build its model
    n.optimize.fix_optimal_capacities()
    n.optimize.create_model()

    # Match the snapshot index of the new network to the base network
    m.set_snapshots(n.snapshots)

    # Select all battery-related stores (case-insensitive)
    battery_stores = n.stores[n.stores.carrier.str.contains("battery", case=False, na=False)]
    battery_buses = battery_stores.bus.unique()

    # Select all battery-related links (chargers/dischargers)
    link_mask = n.links.carrier.str.contains("battery", case=False, na=False)
    battery_links = n.links[link_mask]

    # Collect all buses referenced by those stores and links
    store_buses = battery_stores.bus.tolist()
    link_buses = pd.concat([battery_links.bus0, battery_links.bus1,
                            battery_links.bus2, battery_links.bus3]).dropna().tolist()

    # Unique, non-empty bus names that must be imported
    needed_buses = list(set(store_buses + link_buses))
    needed_buses = [b for b in needed_buses if isinstance(b, str) and b.strip()]

    # Import only the required components into the reduced network
    m.import_components_from_dataframe(n.buses.loc[needed_buses], "Bus")
    m.import_components_from_dataframe(battery_stores, "Store")
    m.import_components_from_dataframe(battery_links, "Link")

    # Add an unlimited, zero-cost source so the storage can charge
    m.add("Generator", "source", bus="DE0 0", p_nom=10e15, marginal_cost=0, efficiency=1)

    # Add a constant load so the storage can discharge
    m.add("Load", "fixed_load", bus="DE0 0", p_set=5e9)

    # Disable cyclic state of charge over the full horizon and build the model
    m.stores.e_cyclic = False
    # m.stores.e_cyclic_per_period = False

    m.optimize.create_model()

    return m

### Extra Functionality for the Optimization

# Build a callback that sets a stochastic profit-maximization objective over given price scenarios
def change_obj(price_list):

    def helper(k, sns):
        # For debugging/inspection, show the last snapshot
        print(sns[-1])

        # Power-flow variables (charger/discharger) for two battery vintages
        battery_charger_2035   = k.model.variables["Link-p"].sel({"Link": "DE0 0 battery charger-2035"})
        battery_discharger_2035= k.model.variables["Link-p"].sel({"Link": "DE0 0 battery discharger-2035"})

        battery_charger_2030   = k.model.variables["Link-p"].sel({"Link": "DE0 0 battery charger-2030"})
        battery_discharger_2030= k.model.variables["Link-p"].sel({"Link": "DE0 0 battery discharger-2030"})

        # Round-trip components (charging efficiencies retrieved for completeness)
        efficiency_2035 = k.links.efficiency["DE0 0 battery charger-2035"]
        efficiency_2030 = k.links.efficiency["DE0 0 battery charger-2030"]

        # Marginal cost add-ons (charging/discharging spreads)
        mc_30 = 0.03
        mc_35 = 0.02

        # Terminal energy value: total stored energy at the last snapshot
        capacity_battery_end = (
            k.model.variables["Store-e"].sel({"Store": "DE0 0 battery-2035", "snapshot": sns[-1]})
            + k.model.variables["Store-e"].sel({"Store": "DE0 0 battery-2030", "snapshot": sns[-1]})
        )

        # Expected profit across price scenarios (equal weights)
        expected_profit = 0
        for price in price_list:
            # Align the price series with the model snapshots
            price_xr = xr.DataArray(price.values, coords={"snapshot": price.index}, dims="snapshot")

            # Terminal price floor used for valuing end-of-horizon energy
            # (note: earlier variant computed a data-driven floor; here it is fixed)
            last_time = sns[-1]
            # (kept for reference; overridden below)
            if (last_time + pd.Timedelta(hours=24)).year > 2019:
                forecast_min = 0
            else:
                forecast_min = price[last_time : last_time + pd.Timedelta(hours=24)].min()
            forecast_min = 50  # previously 10

            # Discharge revenue minus charge cost, plus terminal value; averaged over scenarios
            expected_profit += (
                ( battery_discharger_2030 * (price_xr - mc_30) * efficiency_2030
                + battery_discharger_2035 * (price_xr - mc_35) * efficiency_2035
                - battery_charger_2030   * (price_xr + mc_30)
                - battery_charger_2035   * (price_xr + mc_35)
                ).sum()
                + capacity_battery_end * forecast_min
            ) / len(price_list)

        # Set the objective to maximize expected profit
        k.model.objective = expected_profit
        k.model.sense = 'max'

    return helper


In [15]:
# Compute total battery arbitrage profit for a network over a given time window.
# The profit sums contributions from 2035-vintage and 2030-vintage battery assets.
# 'price' is a time series (aligned with network snapshots) of market prices.
def get_profit(n, price, time_zone=slice('2019-01-01 00:00:00', '2019-12-31 23:59:00')):

    # Power flows (positive sign conventions follow PyPSA): charger/discharger links
    discharge_2035 = n.links_t.p0["DE0 0 battery discharger-2035"]
    charge_2035    = n.links_t.p0["DE0 0 battery charger-2035"]
    discharge_2030 = n.links_t.p0["DE0 0 battery discharger-2030"]
    charge_2030    = n.links_t.p0["DE0 0 battery charger-2030"]

    # Marginal cost add-ons (€/MWh-equivalent) for 2035 and 2030 tech
    mc_35 = 0.02
    mc_30 = 0.03


    # Profit from 2035 battery:
    #   - Charging incurs a cost: charge * (price + mc_35)
    #   - Discharging yields revenue reduced by mc_35 and scaled by charging efficiency
    expected_profit_35 = (-charge_2035[time_zone] * (price[time_zone] + mc_35)  
                          + discharge_2035[time_zone] * n.links.efficiency["DE0 0 battery charger-2035"] * (price[time_zone] - mc_35)).sum()

    # Profit from 2030 battery (analogous structure, using 2030 parameters)
    expected_profit_30 = (-charge_2030[time_zone] * (price[time_zone] + mc_30)
                          + discharge_2030[time_zone] * n.links.efficiency["DE0 0 battery charger-2030"] * (price[time_zone] - mc_35)).sum()

    # Total profit over the specified time slice
    return (expected_profit_35 + expected_profit_30)


In [10]:
# retrieve the prices from the networks with the forecasts

with open("prices.pkl", "rb") as f:
    price_list = pickle.load(f)

real_price = n_base.buses_t.marginal_price['DE0 0']

In [75]:
# optimize stochastically with overlap

n_stochastic = prepare_profit_optimization_network()

n_stochastic.optimize.optimize_with_rolling_horizon(
    solver_name = "gurobi",
    extra_functionality = change_obj(price_list),
    horizon = horizon,
    overlap = overlap
)

INFO:pypsa.io:Imported network base_s_1__none_2035_lt.nc has buses, carriers, generators, global_constraints, links, loads, storage_units, stores
{'DE', 'EU'}
INFO:pypsa.optimization.abstract:Optimizing network for snapshot horizon [2019-01-01 00:00:00:2019-01-06 21:00:00] (1/61).


2019-01-06 21:00:00


INFO:linopy.model: Solve problem using Gurobi solver


Set parameter Username


INFO:gurobipy:Set parameter Username


...

INFO:gurobipy:


Solved in 12 iterations and 0.01 seconds (0.00 work units)


INFO:gurobipy:Solved in 12 iterations and 0.01 seconds (0.00 work units)


Optimal objective  7.266898025e+06


INFO:gurobipy:Optimal objective  7.266898025e+06
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 440 primals, 880 duals
Objective: 7.27e+06
Solver model: available
Solver message: 2

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper, Store-fix-e-lower, Store-fix-e-upper, Store-energy_balance were not assigned to the network.


In [76]:
# optimize deterministically with overlap

networks_deterministic = []

for i in range(0, nr_networks):
    n = prepare_profit_optimization_network()

    n.optimize.optimize_with_rolling_horizon(
        solver_name = "gurobi",
        extra_functionality = change_obj([price_list[i]]),
        horizon = horizon,
        overlap = overlap,
    )

    networks_deterministic.append(n)

INFO:pypsa.io:Imported network base_s_1__none_2035_lt.nc has buses, carriers, generators, global_constraints, links, loads, storage_units, stores
{'DE', 'EU'}
INFO:pypsa.optimization.abstract:Optimizing network for snapshot horizon [2019-01-01 00:00:00:2019-01-06 21:00:00] (1/61).
INFO:linopy.model: Solve problem using Gurobi solver


2019-01-06 21:00:00
Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2637156


INFO:gurobipy:Set parameter LicenseID to value 2637156


...

INFO:gurobipy:


Solved in 14 iterations and 0.02 seconds (0.00 work units)


INFO:gurobipy:Solved in 14 iterations and 0.02 seconds (0.00 work units)


Optimal objective  2.454907658e+07


INFO:gurobipy:Optimal objective  2.454907658e+07
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 440 primals, 880 duals
Objective: 2.45e+07
Solver model: available
Solver message: 2

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper, Store-fix-e-lower, Store-fix-e-upper, Store-energy_balance were not assigned to the network.


In [77]:
# print the profits

profit = print_profits(time_zone=slice('2019-01-01 00:00:00', '2019-12-31 23:59:00'))

Profit in Mrd. €


,n0,n1,n2,n3,n4,n5,n_stochastic
price 1,2.64,2.11,2.22,2.05,2.19,2.16,2.50
price 2,2.35,2.72,2.32,2.39,2.35,2.31,2.63
price 3,2.19,2.03,2.74,2.15,2.18,2.18,2.55
price 4,2.09,2.23,2.17,2.68,2.23,2.22,2.47
price 5,2.62,2.37,2.25,2.34,3.07,2.44,2.83
price 6,2.77,2.79,2.82,2.82,2.64,3.30,3.11
real_price,1.67,1.67,1.61,1.78,1.73,1.64,1.88
mean,2.44,2.38,2.42,2.41,2.44,2.44,2.68
